# Step 1: Load the data from all CSV files in the directory and concatenate them into a single DataFrame.

In [1]:
import os
import pandas as pd

directory_path = 'ncaa_box_scores_2024_25'

dataframes = []

for file_name in os.listdir(directory_path):
    file_path = os.path.join(directory_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

final_dataframe = pd.concat(dataframes, ignore_index=True)

final_dataframe['team_score'] = final_dataframe['team_score'] / 2
final_dataframe['opponent_score'] = final_dataframe['opponent_score'] / 2

final_dataframe

,player_id,player,MIN,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,...,team,opponent,home,starter,game_id,date,location,team_score,opponent_score,result
0,5177717.0,M. Gomma,26.0,3,4,0,0,1,2,1,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
1,4610395.0,M. Moncrieffe,36.0,7,9,0,0,6,10,3,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
2,5241342.0,B. Maldonado,14.0,2,9,2,5,1,1,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
3,5107830.0,M. Arington,31.0,3,7,1,1,1,1,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
4,4433606.0,J. Christofilis,31.0,4,11,4,6,5,6,0,...,Seattle U,Eastern Washington,False,True,401718465,2024-11-06,A,86.0,93.0,L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131998,5106735.0,J. Karasinski,15.0,2,6,1,3,0,0,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L
131999,5239765.0,M. Watkins,23.0,6,10,1,3,0,0,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L
132000,5175849.0,AJ Lux,23.0,3,5,2,4,0,0,1,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L
132001,4702669.0,D. McKinney,7.0,2,3,1,2,2,2,0,...,Bellarmine,Florida Gulf Coast,False,False,401711649,2025-02-26,A,61.0,80.0,L


# Step 2: Clean the data by removing player data and keeping only team data. In the process, we drop unnecessary columns and reformmated the date column.

In [2]:
team_dataframe = final_dataframe[(pd.isna(final_dataframe['player_id'])) & (final_dataframe['player'] == 'TEAM')]

In [3]:
team_dataframe = team_dataframe.drop(columns=['player_id', 'player','MIN'])
# team_dataframe = team_dataframe.drop_duplicates(subset='game_id', keep='first')

team_dataframe['date'] = pd.to_datetime(team_dataframe['date'])
team_dataframe = team_dataframe.sort_values(by='date', ascending=True)

team_dataframe

,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,DREB,REB,AST,...,team,opponent,home,starter,game_id,date,location,team_score,opponent_score,result
36619,30,67,6,19,7,7,7,20,27,14,...,Texas A&M-Corpus Christi,Purdue,False,False,401707976,2024-11-04,A,73.0,90.0,L
125064,22,51,10,26,10,13,5,20,25,12,...,Canisius,Arizona,False,False,401706149,2024-11-04,A,64.0,93.0,L
91215,22,58,10,27,13,16,8,27,35,11,...,Vermont,UAB,False,False,401722181,2024-11-04,A,67.0,62.0,W
90863,17,58,5,24,5,15,14,21,35,7,...,Eastern Michigan,Texas State,False,False,401721019,2024-11-04,A,44.0,64.0,L
15622,32,57,10,22,9,17,5,24,29,17,...,Buffalo,Old Dominion,False,False,401721009,2024-11-04,A,83.0,82.0,W
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114455,21,53,7,17,18,23,10,21,31,14,...,Duke,Houston,True,False,401746080,2025-04-05,N,67.0,70.0,L
82061,29,61,9,21,10,17,9,29,38,20,...,Nebraska,UCF,False,False,401758636,2025-04-06,N,77.0,66.0,W
27602,27,68,7,29,5,6,13,26,39,8,...,UCF,Nebraska,True,False,401758636,2025-04-06,N,66.0,77.0,L
101631,21,53,6,24,17,21,8,32,40,14,...,Florida,Houston,False,False,401746082,2025-04-07,N,65.0,63.0,W


In [4]:
print(team_dataframe.columns)

Index(['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB',
       'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'team', 'opponent', 'home',
       'starter', 'game_id', 'date', 'location', 'team_score',
       'opponent_score', 'result'],
      dtype='object')


# step 3: combine 2 rows with the same game id into 1

In [5]:
team_df = team_dataframe.copy()
opp_df = team_dataframe.copy()

team_df = team_df.rename(columns={col: f'team_{col}' for col in ['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'FG_pct', '3PT_pct', 'FT_pct', 'AST_TO_ratio', 'TS_pct', 'team', 'team_score', 'result']})
opp_df = opp_df.rename(columns={col: f'opp_{col}' for col in ['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'FG_pct', '3PT_pct', 'FT_pct', 'AST_TO_ratio', 'TS_pct', 'team', 'team_score', 'result']})

matchup_df = pd.merge(team_df, opp_df, on='game_id')

matchup_df = matchup_df[matchup_df['team_team'] != matchup_df['opp_team']]

In [6]:
team_dataframe['game_pair_id'] = team_dataframe['game_id'].astype(str) + '_' + team_dataframe[['team', 'opponent']].min(axis=1) + '_' + team_dataframe[['team', 'opponent']].max(axis=1)


In [7]:
team_stats = team_dataframe.copy()
opp_stats = team_dataframe.copy()

# Rename columns for opponent
opp_stats = opp_stats.add_prefix('opp_')
opp_stats.rename(columns={'opp_game_pair_id': 'game_pair_id'}, inplace=True)

# Merge
matchups = pd.merge(team_stats, opp_stats, on='game_pair_id')

# Filter: keep only rows where team != opponent (avoid self-match)
matchups = matchups[matchups['team'] != matchups['opp_team']]


In [22]:
# Drop duplicates
matchups = matchups[matchups['team'] < matchups['opp_team']]

matchups

Index(['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB',
       'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'team', 'opponent', 'home',
       'starter', 'game_id', 'date', 'location', 'team_score',
       'opponent_score', 'result', 'game_pair_id', 'opp_FGM', 'opp_FGA',
       'opp_3PTM', 'opp_3PTA', 'opp_FTM', 'opp_FTA', 'opp_OREB', 'opp_DREB',
       'opp_REB', 'opp_AST', 'opp_STL', 'opp_BLK', 'opp_TO', 'opp_PF',
       'opp_PTS', 'opp_team', 'opp_opponent', 'opp_home', 'opp_starter',
       'opp_game_id', 'opp_date', 'opp_location', 'opp_team_score',
       'opp_opponent_score', 'opp_result', 'fg_pct_diff', '3pt_pct_diff',
       'ft_pct_diff', 'reb_diff', 'ast_diff', 'to_diff', 'pts_diff',
       'Possession_proxies', 'label'],
      dtype='object')


# step 4: Create new features for the model. These features are the differences between the two teams' stats.

In [9]:
matchups['fg_pct_diff'] = (matchups['FGM'] / matchups['FGA']) - (matchups['opp_FGM'] / matchups['opp_FGA'])
matchups['3pt_pct_diff'] = (matchups['3PTM'] / matchups['3PTA']) - (matchups['opp_3PTM'] / matchups['opp_3PTA'])
matchups['ft_pct_diff'] = (matchups['FTM'] / matchups['FTA']) - (matchups['opp_FTM'] / matchups['opp_FTA'])
matchups['reb_diff'] = matchups['REB'] - matchups['opp_REB']
matchups['ast_diff'] = matchups['AST'] - matchups['opp_AST']
matchups['to_diff'] = matchups['TO'] - matchups['opp_TO']
matchups['pts_diff'] = matchups['PTS'] - matchups['opp_PTS']

team_pp = matchups['FGA'] + (0.44 * matchups['FTA']) - matchups['OREB'] + matchups['TO']
opponent_pp = matchups['opp_FGA'] + (0.44 * matchups['opp_FTA']) - matchups['opp_OREB'] + matchups['opp_TO']
matchups['Possession_proxies'] = team_pp / opponent_pp

In [37]:
# matchups['label'] = matchups.apply(
#     lambda row: 1 if row['team_score'] > row['opp_team_score'] else 2, axis=1
# )
matchups['label'] = (matchups['team_score'] > matchups['opp_team_score']).astype(int)

matchups = matchups[[
    'date', 'game_id',
    'team', 'team_score', 'location', 'FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB',
    'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS',
    'opp_team', 'opp_team_score', 'opp_location', 'opp_FGM', 'opp_FGA', 'opp_3PTM', 'opp_3PTA', 'opp_FTM', 'opp_FTA',
    'opp_OREB', 'opp_DREB', 'opp_REB', 'opp_AST', 'opp_STL', 'opp_BLK', 'opp_TO', 'opp_PF', 'opp_PTS',
    'fg_pct_diff', '3pt_pct_diff', 'ft_pct_diff', 'reb_diff', 'ast_diff', 'to_diff', 'pts_diff',
    'Possession_proxies', 'label'
]]

matchups

,date,game_id,team,team_score,location,FGM,FGA,3PTM,3PTA,FTM,...,opp_PTS,fg_pct_diff,3pt_pct_diff,ft_pct_diff,reb_diff,ast_diff,to_diff,pts_diff,Possession_proxies,label
2,2024-11-04,401707976,Purdue,90.0,H,31,55,11,26,17,...,73,0.115875,0.107287,-0.370370,12,10,7,17,1.011099,1
6,2024-11-04,401706149,Arizona,93.0,H,33,68,8,28,19,...,64,0.053922,-0.098901,-0.065527,18,8,-10,29,1.016403,1
10,2024-11-04,401722181,UAB,62.0,H,26,68,0,10,10,...,67,0.003043,-0.370370,-0.145833,14,-8,2,-5,1.025145,0
13,2024-11-04,401721019,Eastern Michigan,44.0,A,17,58,5,24,5,...,64,-0.123563,-0.258333,-0.476190,-8,-6,0,-20,1.037918,0
17,2024-11-04,401721009,Buffalo,83.0,A,32,57,10,22,9,...,82,0.132832,0.054545,0.029412,-17,2,-2,1,0.985854,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,2025-04-05,401746081,Auburn,73.0,N,25,57,7,25,16,...,79,-0.033102,-0.100952,-0.004348,-9,4,-2,-6,1.027350,0
23264,2025-04-05,401758633,UCF,104.0,N,30,63,15,35,29,...,98,0.073413,0.170507,-0.139865,-3,2,0,6,0.965764,1
23268,2025-04-05,401758632,Boise State,69.0,N,25,63,9,27,10,...,79,-0.067460,-0.298246,0.075758,-7,-4,-2,-10,1.030038,0
23272,2025-04-06,401758636,Nebraska,77.0,N,29,61,9,21,10,...,66,0.078351,0.187192,-0.245098,-1,12,-3,11,0.983343,1


# Step 5: split the data for trainign and testing after x date

In [38]:
matchups['date'] = pd.to_datetime(matchups['date'])

split_date = pd.Timestamp('2025-02-16')
train_df = matchups[matchups['date'] < split_date]
test_df = matchups[matchups['date'] >= split_date]

In [39]:
train_df

,date,game_id,team,team_score,location,FGM,FGA,3PTM,3PTA,FTM,...,opp_PTS,fg_pct_diff,3pt_pct_diff,ft_pct_diff,reb_diff,ast_diff,to_diff,pts_diff,Possession_proxies,label
2,2024-11-04,401707976,Purdue,90.0,H,31,55,11,26,17,...,73,0.115875,0.107287,-0.370370,12,10,7,17,1.011099,1
6,2024-11-04,401706149,Arizona,93.0,H,33,68,8,28,19,...,64,0.053922,-0.098901,-0.065527,18,8,-10,29,1.016403,1
10,2024-11-04,401722181,UAB,62.0,H,26,68,0,10,10,...,67,0.003043,-0.370370,-0.145833,14,-8,2,-5,1.025145,0
13,2024-11-04,401721019,Eastern Michigan,44.0,A,17,58,5,24,5,...,64,-0.123563,-0.258333,-0.476190,-8,-6,0,-20,1.037918,0
17,2024-11-04,401721009,Buffalo,83.0,A,32,57,10,22,9,...,82,0.132832,0.054545,0.029412,-17,2,-2,1,0.985854,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17985,2025-02-15,401721246,Alabama A&M,82.0,A,24,52,8,20,26,...,70,-0.061189,-0.016667,0.108453,12,4,-3,12,0.974475,1
17990,2025-02-15,401724433,Massachusetts,59.0,H,18,51,3,17,20,...,73,-0.128540,-0.131222,0.001610,-6,-3,0,-14,1.010839,0
17994,2025-02-15,401721186,Mercyhurst,73.0,A,23,53,6,12,21,...,85,-0.121593,0.136364,-0.072222,-15,-5,-2,-12,1.034211,0
17998,2025-02-15,401714598,Arkansas State,70.0,H,25,65,8,26,12,...,71,-0.001350,0.021978,-0.037037,8,-1,3,-1,1.015321,0


In [40]:
test_df

,date,game_id,team,team_score,location,FGM,FGA,3PTM,3PTA,FTM,...,opp_PTS,fg_pct_diff,3pt_pct_diff,ft_pct_diff,reb_diff,ast_diff,to_diff,pts_diff,Possession_proxies,label
18006,2025-02-16,401706268,Iona,74.0,A,28,65,11,28,7,...,79,-0.102564,0.039916,0.125000,-16,-2,-8,-5,0.990166,0
18010,2025-02-16,401721435,Oregon,75.0,H,28,60,13,22,6,...,57,0.005128,0.380383,0.035714,6,11,-4,18,0.990979,1
18014,2025-02-16,401721434,Michigan,86.0,A,33,68,7,21,13,...,83,0.026278,-0.026667,0.098039,15,0,3,3,0.979966,1
18019,2025-02-16,401714455,Cleveland State,60.0,H,22,62,3,20,13,...,68,-0.111828,-0.278571,-0.085020,-6,0,-7,-8,1.030750,0
18022,2025-02-16,401714452,Milwaukee,92.0,H,32,59,7,19,21,...,70,0.097928,0.050239,0.027778,24,8,5,22,1.018367,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,2025-04-05,401746081,Auburn,73.0,N,25,57,7,25,16,...,79,-0.033102,-0.100952,-0.004348,-9,4,-2,-6,1.027350,0
23264,2025-04-05,401758633,UCF,104.0,N,30,63,15,35,29,...,98,0.073413,0.170507,-0.139865,-3,2,0,6,0.965764,1
23268,2025-04-05,401758632,Boise State,69.0,N,25,63,9,27,10,...,79,-0.067460,-0.298246,0.075758,-7,-4,-2,-10,1.030038,0
23272,2025-04-06,401758636,Nebraska,77.0,N,29,61,9,21,10,...,66,0.078351,0.187192,-0.245098,-1,12,-3,11,0.983343,1


# Step 6: train the data

In [41]:
# feature_cols = [
#     'fg_pct_diff', '3pt_pct_diff', 'ft_pct_diff', 
#     'reb_diff', 'ast_diff', 'to_diff', 'pts_diff', 'Possession_proxies'
# ]

feature_cols = [
    'fg_pct_diff', '3pt_pct_diff', 'ft_pct_diff', 
    'reb_diff', 'ast_diff', 'to_diff', 'pts_diff',
]

X_train = train_df[feature_cols]
X_train = X_train.fillna(0)
y_train = train_df['label']

In [42]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

#model accuracy

RandomForestClassifier(random_state=42)

# Step 7: Evaluate the model on the test set

In [43]:
# Copy and restructure the DataFrame
latest_team_dataframe = team_dataframe.copy()

# Select and reorder the specified columns
latest_team_dataframe = latest_team_dataframe[
    ['date', 'game_id', 'team', 'FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 
     'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'home', 'team_score']
]

latest_team_dataframe

,date,game_id,team,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,home,team_score
36619,2024-11-04,401707976,Texas A&M-Corpus Christi,30,67,6,19,7,7,7,20,27,14,10,0,9,25,73,False,73.0
125064,2024-11-04,401706149,Canisius,22,51,10,26,10,13,5,20,25,12,7,1,19,20,64,False,64.0
91215,2024-11-04,401722181,Vermont,22,58,10,27,13,16,8,27,35,11,4,4,5,14,67,False,67.0
90863,2024-11-04,401721019,Eastern Michigan,17,58,5,24,5,15,14,21,35,7,7,0,14,15,44,False,44.0
15622,2024-11-04,401721009,Buffalo,32,57,10,22,9,17,5,24,29,17,9,6,13,10,83,False,83.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114455,2025-04-05,401746080,Duke,21,53,7,17,18,23,10,21,31,14,7,4,7,16,67,True,67.0
82061,2025-04-06,401758636,Nebraska,29,61,9,21,10,17,9,29,38,20,8,3,9,14,77,False,77.0
27602,2025-04-06,401758636,UCF,27,68,7,29,5,6,13,26,39,8,7,5,12,17,66,True,66.0
101631,2025-04-07,401746082,Florida,21,53,6,24,17,21,8,32,40,14,7,5,13,11,65,False,65.0


In [44]:
rows = []

schedule_df = pd.read_csv('schedule.csv')

def get_latest_game(date, team):
    # Filter games where the team is either home or away, game_id is less than max_game_id, and date is before the given date
    team_games = schedule_df[
        ((schedule_df['home'] == team) | (schedule_df['away'] == team)) &
        (pd.to_datetime(schedule_df['date']) < pd.to_datetime(date))
    ]
    
    # Sort by game_id in descending order and get the latest game
    latest_game = team_games.sort_values('game_id', ascending=False).head(1)
    return latest_game['game_id'].values[0] if not latest_game.empty else None


for index, row in test_df.iterrows():
    latest_home_game = get_latest_game(row['date'], row['team'])
    latest_away_game = get_latest_game(row['date'], row['opp_team'])

    print(f"Teams playing in date {row['date']}: {row['team']} vs {row['opp_team']}")
    print(f"Latest game_id for {row['team']} before {row['date']}: {latest_home_game}")
    print(f"Latest game_id for {row['opp_team']} before {row['date']}: {latest_away_game}")
    
    # Filter the DataFrame to find the row with the matching game_id
    row_home = latest_team_dataframe[latest_team_dataframe['game_id'] == latest_home_game]
    row_opp = latest_team_dataframe[latest_team_dataframe['game_id'] == latest_away_game]
    
    fg_pct_diff = row_home['FGM'].values[0] / row_home['FGA'].values[0] - row_opp['FGM'].values[0] / row_opp['FGA'].values[0]
    three_pt_pct_diff = row_home['3PTM'].values[0] / row_home['3PTA'].values[0] - row_opp['3PTM'].values[0] / row_opp['3PTA'].values[0]
    ft_pct_diff = row_home['FTM'].values[0] / row_home['FTA'].values[0] - row_opp['FTM'].values[0] / row_opp['FTA'].values[0]
    reb_diff = row_home['REB'].values[0] - row_opp['REB'].values[0]
    ast_diff = row_home['AST'].values[0] - row_opp['AST'].values[0]
    to_diff = row_home['TO'].values[0] - row_opp['TO'].values[0]
    pts_diff = row_home['PTS'].values[0] - row_opp['PTS'].values[0]

    team_pp = row_home['FGA'].values[0] + (0.44 * row_home['FTA'].values[0]) - row_home['OREB'].values[0] + row_home['TO'].values[0]
    opponent_pp = row_opp['FGA'].values[0] + (0.44 * row_opp['FTA'].values[0]) - row_opp['OREB'].values[0] + row_opp['TO'].values[0]
    possession_proxies = team_pp / opponent_pp

    # Append the calculated row to the list
    rows.append({
        'fg_pct_diff': fg_pct_diff,
        '3pt_pct_diff': three_pt_pct_diff,
        'ft_pct_diff': ft_pct_diff,
        'reb_diff': reb_diff,
        'ast_diff': ast_diff,
        'to_diff': to_diff,
        'pts_diff': pts_diff,
        'Possession_proxies': possession_proxies
    })
        
    print("----------------------------------")

testing_df = pd.DataFrame(rows)
testing_df

# matchups['fg_pct_diff'] = (matchups['FGM'] / matchups['FGA']) - (matchups['opp_FGM'] / matchups['opp_FGA'])
# matchups['3pt_pct_diff'] = (matchups['3PTM'] / matchups['3PTA']) - (matchups['opp_3PTM'] / matchups['opp_3PTA'])
# matchups['ft_pct_diff'] = (matchups['FTM'] / matchups['FTA']) - (matchups['opp_FTM'] / matchups['opp_FTA'])
# matchups['reb_diff'] = matchups['REB'] - matchups['opp_REB']
# matchups['ast_diff'] = matchups['AST'] - matchups['opp_AST']
# matchups['to_diff'] = matchups['TO'] - matchups['opp_TO']
# matchups['pts_diff'] = matchups['PTS'] - matchups['opp_PTS']
# 
# team_pp = matchups['FGA'] + (0.44 * matchups['FTA']) - matchups['OREB'] + matchups['TO']
# opponent_pp = matchups['opp_FGA'] + (0.44 * matchups['opp_FTA']) - matchups['opp_OREB'] + matchups['opp_TO']
# matchups['Possession_proxies'] = team_pp - opponent_pp

Teams playing in date 2025-02-16 00:00:00: Iona vs Quinnipiac
Latest game_id for Iona before 2025-02-16 00:00:00: 401727310.0
Latest game_id for Quinnipiac before 2025-02-16 00:00:00: 401727307.0
----------------------------------
Teams playing in date 2025-02-16 00:00:00: Oregon vs Rutgers
Latest game_id for Oregon before 2025-02-16 00:00:00: 401730560.0
Latest game_id for Rutgers before 2025-02-16 00:00:00: 401730558.0
----------------------------------
Teams playing in date 2025-02-16 00:00:00: Michigan vs Ohio State
Latest game_id for Michigan before 2025-02-16 00:00:00: 401729802.0
Latest game_id for Ohio State before 2025-02-16 00:00:00: 401721424.0
----------------------------------
Teams playing in date 2025-02-16 00:00:00: Cleveland State vs Youngstown State
Latest game_id for Cleveland State before 2025-02-16 00:00:00: 401727283.0
Latest game_id for Youngstown State before 2025-02-16 00:00:00: 401740670.0
----------------------------------
Teams playing in date 2025-02-16 00:

,fg_pct_diff,3pt_pct_diff,ft_pct_diff,reb_diff,ast_diff,to_diff,pts_diff,Possession_proxies
0,-0.024276,0.000000,-0.089592,-15,-5,2,-8,0.860746
1,0.027629,0.063348,-0.082353,-3,4,-8,6,0.954498
2,-0.236991,-0.263285,-0.262626,4,-8,5,-40,1.018370
3,0.056209,0.112701,0.232143,-21,5,-5,4,0.996077
4,0.131313,0.052174,-0.019139,-5,4,-2,34,1.354798
...,...,...,...,...,...,...,...,...
1303,0.000000,0.000000,0.000000,0,0,0,0,1.000000
1304,-0.039171,0.040441,-0.046218,-2,5,0,-4,0.904832
1305,0.183448,0.195175,0.093333,-1,10,-12,39,0.942108
1306,-0.079365,-0.095238,0.184091,-10,0,2,-35,0.870899


In [45]:
feature_cols = [
    'fg_pct_diff', '3pt_pct_diff', 'ft_pct_diff', 'reb_diff', 'ast_diff', 'to_diff', 'pts_diff', 'Possession_proxies'
]

X_train = train_df[feature_cols]
X_train = X_train.fillna(0)
y_train = train_df['label']

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

X_test = testing_df[feature_cols]
X_test = X_test.fillna(0)
y_test = test_df['label']

y_pred = model.predict(X_test)
y_pred

array([0, 1, 0, ..., 1, 0, 1])

In [46]:
from sklearn.metrics import accuracy_score, classification_report

# Step 7: Evaluate
print("🧪 Model Evaluation After 2025-02-16")
print("----------------------------------")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

🧪 Model Evaluation After 2025-02-16
----------------------------------
Accuracy: 0.5076

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.49      0.49       638
           1       0.52      0.53      0.52       670

    accuracy                           0.51      1308
   macro avg       0.51      0.51      0.51      1308
weighted avg       0.51      0.51      0.51      1308


In [20]:
# Step 8: Create a results DataFrame
# Get prediction probabilities
y_proba = model.predict_proba(X_test)

# Get the probability of the predicted class (highest prob per row)
confidence = y_proba.max(axis=1)

# Add predicted label, confidence, actual label, and correctness
results = test_df.copy()
results['predicted_label'] = y_pred
results['confidence'] = confidence
results['actual_label'] = y_test.values
results['correct'] = results['predicted_label'] == results['actual_label']
results['status'] = results['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')
results


,FGM,FGA,3PTM,3PTA,FTM,FTA,OREB,DREB,REB,AST,...,ast_diff,to_diff,pts_diff,Possession_proxies,label,predicted_label,confidence,actual_label,correct,status
18006,28,65,11,28,7,8,5,17,22,15,...,-2,-8,-5,0.990166,0,0,1.000000,0,True,✅ Correct
18010,28,60,13,22,6,8,11,22,33,16,...,11,-4,18,0.990979,1,1,0.970000,1,True,✅ Correct
18014,33,68,7,21,13,17,19,27,46,13,...,0,3,3,0.979966,1,0,1.000000,1,False,❌ Wrong
18019,22,62,3,20,13,19,17,15,32,10,...,0,-7,-8,1.030750,0,1,0.910000,0,False,❌ Wrong
18022,32,59,7,19,21,27,21,22,43,18,...,8,5,22,1.018367,1,1,0.990000,1,True,✅ Correct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23261,25,57,7,25,16,23,9,21,30,11,...,4,-2,-6,1.027350,0,1,0.613472,0,False,❌ Wrong
23264,30,63,15,35,29,40,12,29,41,14,...,2,0,6,0.965764,1,0,0.990000,1,False,❌ Wrong
23268,25,63,9,27,10,11,11,20,31,14,...,-4,-2,-10,1.030038,0,1,0.990000,0,False,❌ Wrong
23272,29,61,9,21,10,17,9,29,38,20,...,12,-3,11,0.983343,1,0,1.000000,1,False,❌ Wrong


In [21]:


# Step 9: Show key columns
display_cols = ['date', 'team', 'opponent', 'predicted_label', 'confidence', 'actual_label', 'status']
print("\n📋 Prediction Results with Confidence:")
print(results[display_cols].sort_values('date').to_string(index=False))


📋 Prediction Results with Confidence:
      date                     team                 opponent  predicted_label  confidence  actual_label    status
2025-02-16                     Iona               Quinnipiac                0    1.000000             0 ✅ Correct
2025-02-16                 Nebraska             Northwestern                1    0.980000             1 ✅ Correct
2025-02-16                    Rider            Saint Peter's                1    1.000000             0   ❌ Wrong
2025-02-16               Evansville               Valparaiso                0    1.000000             1   ❌ Wrong
2025-02-16                   Marist                Merrimack                0    0.960000             1   ❌ Wrong
2025-02-16               Louisville               Notre Dame                0    1.000000             1   ❌ Wrong
2025-02-16            South Florida                      UAB                0    0.990000             0 ✅ Correct
2025-02-16                Fairfield              